# Fig 3 export drop-in: per-sample standardized scores (CIFAR-10)

Fig. 3 plots the distribution of standardized scores for clean / benign-noise / adversarial inputs,
for HF-Energy (left) and the median aggregate (right). Those are per-sample values, which the summary
AUROCs do not contain, so this notebook exports them. Run it, then upload
`fig3_results/fig3_scores.json` and I will render the violin plot.

Same conventions as the other drop-ins: [0,255] images, `make_pp`, `load_backbone`, `feat_*`,
`mixed_dataset.pkl`. Standardization stats (mu, sigma) come from the clean calibration half, exactly
as in the centerpiece harness, so HF-Energy's benign-noise z-scores will land very high (matching the
"benign noise at the top" story) and the median aggregate will pull them back down.


In [ ]:
# ===================== [PREAMBLE] (skip if reusing your kernel) =====================
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
def make_pp(ds):
    mean=torch.tensor(CIFAR_MEAN).view(1,3,1,1).to(device); std=torch.tensor(CIFAR_STD).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
    m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
    m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75): return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    return p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9).median(-1).values
def feat_hfe(b): return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()
def feat_gl(b,bb,pp,glsig):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(b,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
    return (p0-p2).abs().sum(1).cpu().numpy()
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
    return out
print('[PREAMBLE] ready')


In [ ]:
# ===================== export per-sample z-scores (CIFAR-10) =====================
SEED=42; glsig=0.5
def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]

bb=load_backbone('CIFAR-10'); pp=make_pp('CIFAR-10')
mixed=pickle.load(open(find_mixed()['CIFAR-10'],'rb'))
clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
adv  =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
rng=np.random.RandomState(SEED); clean=[clean[i] for i in rng.permutation(len(clean))[:500]]
Xc=torch.cat(clean,0); Xa=torch.cat(adv,0)
ci,ti=half(len(clean)); Xcal=Xc[ci]; Xte=Xc[ti]
noise=(Xte+torch.randn_like(Xte)*8.0).clamp(0,255)   # matched-budget benign noise

def feats(X):
    H=[];G=[];P=[]
    for i in range(0,len(X),64):
        b=X[i:i+64].to(device); H.append(feat_hfe(b)); G.append(feat_gl(b,bb,pp,glsig)); P.append(feat_predl1(b,bb,pp))
    return np.concatenate(H),np.concatenate(G),np.concatenate(P)

# standardization stats from clean calibration half
Hc,Gc,Pc=feats(Xcal)
muH,sdH=Hc.mean(),Hc.std()+1e-8; muG,sdG=Gc.mean(),Gc.std()+1e-8; muP,sdP=Pc.mean(),Pc.std()+1e-8

def zset(X):
    H,G,P=feats(X)
    zH=(H-muH)/sdH; zG=(G-muG)/sdG; zP=(P-muP)/sdP
    med=np.median(np.stack([zH,zG,zP],1),1)
    return zH.tolist(), med.tolist()

zH_clean,med_clean=zset(Xte)
zH_noise,med_noise=zset(noise)
zH_adv,  med_adv  =zset(Xa)

OUT='./fig3_results'; os.makedirs(OUT,exist_ok=True)
data={'hf_z':{'clean':zH_clean,'benign_noise':zH_noise,'adversarial':zH_adv},
      'median_agg':{'clean':med_clean,'benign_noise':med_noise,'adversarial':med_adv},
      'meta':{'n_clean':len(zH_clean),'n_noise':len(zH_noise),'n_adv':len(zH_adv),
              'note':'z-scores standardized on clean calibration half; matched-noise sigma=8 on [0,255]'}}
json.dump(data, open(os.path.join(OUT,'fig3_scores.json'),'w'))
print('saved', os.path.join(OUT,'fig3_scores.json'))
print('HF-Energy z medians  -> clean %.2f | noise %.2f | adv %.2f' % (np.median(zH_clean),np.median(zH_noise),np.median(zH_adv)))
print('Median-agg medians   -> clean %.2f | noise %.2f | adv %.2f' % (np.median(med_clean),np.median(med_noise),np.median(med_adv)))
